# Paragraph chunks of Simple English Wikipedia, embedded on a T4

Builds the paragraph corpus for the article-weighted retriever (`rag/article_weighted.py`). Chunking is
`rag/chunking.py` (pasted below unchanged): sections -> paragraphs -> 40..150 words, the previous paragraph kept as
overlap. Embeddings: all-MiniLM-L6-v2, the model of the existing index. Output on Drive:
`para_chunks.parquet` (metadata + text) and `para_vectors_f16.npy` (row-aligned). Then, on the Mac:
`python rag/build_para_table.py` (sprout-index env) builds the LanceDB table.

Source: wikimedia/wikipedia 20231101.simple (CC BY-SA 4.0).

In [ ]:
!pip install -q -U sentence-transformers datasets pyarrow
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/llm_sft_data/wiki_para'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- rag/chunking.py (unchanged copy) ----
"""Paragraph-aware chunking for Wikipedia text, ported from the structural splitter of the earlier RAG project
(sections -> paragraphs -> merged to [min, max] words, the previous paragraph repeated as overlap). Word counts
instead of tokens; the bounds are smaller because the agent's context window is 2048 tokens."""
import re

SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])")
MIN_WORDS, MAX_WORDS = 40, 150


def is_heading(line):
    """wikimedia/wikipedia keeps section titles as short lines without end punctuation."""
    s = line.strip()
    return 0 < len(s.split()) <= 8 and not s.endswith((".", "!", "?", ":", ";", ","))


def _split_long(text, max_words):
    out, buf = [], []
    for s in SENT.split(text):
        if buf and len(" ".join(buf + [s]).split()) > max_words:
            out.append(" ".join(buf)); buf = []
        buf.append(s)
    if buf:
        if out and len(" ".join(buf).split()) < MIN_WORDS // 2:   # a short tail joins the previous piece
            out[-1] += " " + " ".join(buf)
        else:
            out.append(" ".join(buf))
    return out


def chunk_article(text, min_words=MIN_WORDS, max_words=MAX_WORDS):
    """-> list of (section, chunk_text). Paragraphs are blank-line separated; a heading starts a new section."""
    sections, cur, title = [], [], ""
    for block in re.split(r"\n\s*\n", text or ""):
        lines = [l for l in block.strip().splitlines() if l.strip()]
        if not lines:
            continue
        if len(lines) == 1 and is_heading(lines[0]):
            if cur:
                sections.append((title, cur))
            title, cur = lines[0].strip(), []
            continue
        if is_heading(lines[0]):                       # heading glued to its first paragraph
            if cur:
                sections.append((title, cur))
            title, cur, lines = lines[0].strip(), [], lines[1:]
        cur.append(re.sub(r"\s+", " ", " ".join(lines)).strip())
    if cur:
        sections.append((title, cur))

    chunks = []
    for title, paras in sections:
        buf, prev = [], None
        for p in paras:
            n = len(p.split())
            if n > max_words:                          # overflow: sentence-split, no overlap
                if buf:
                    chunks.append((title, " ".join(buf))); buf = []
                chunks += [(title, c) for c in _split_long(p, max_words)]
                prev = None
            elif len(" ".join(buf).split()) + n < min_words or not buf:
                buf.append(p); prev = p
            else:
                chunks.append((title, " ".join(buf)))
                buf = ([prev] if prev and len(prev.split()) + n <= max_words else []) + [p]
                prev = p
        if buf:
            chunks.append((title, " ".join(buf)))
    return [(t, c) for t, c in chunks if len(c.split()) >= 8]


In [ ]:
from datasets import load_dataset
import pandas as pd, numpy as np, time
MIN_CHARS = 200          # same filters as rag/add_simple_wikipedia.py
ds = load_dataset('wikimedia/wikipedia', '20231101.simple', split='train')
rows = []
for r in ds:
    title, body = r['title'].strip(), r['text']
    if len(body) < MIN_CHARS or title.lower().startswith(('list of', 'category:')):
        continue
    for i, (sec, c) in enumerate(chunk_article(body)):
        rows.append(dict(id=f"simple-{r['id']}#p{i}", article_id=f"simple-{r['id']}", topic=title, section=sec,
                         para_idx=i, search_text=f"{title}: {c}", url=r['url']))
df = pd.DataFrame(rows)
print(len(ds), 'articles ->', len(df), 'chunks;', df.groupby('article_id').size().describe()[['mean', '50%', 'max']].to_dict())
print('words per chunk:', df.search_text.str.split().str.len().describe()[['mean', '50%', 'max']].to_dict())
df.sample(3, random_state=0)[['topic', 'section', 'search_text']]

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda').half()
t0 = time.time()
vec = model.encode(df.search_text.tolist(), batch_size=512, show_progress_bar=True, convert_to_numpy=True)
print(f'{len(df)} chunks in {time.time() - t0:.0f}s')
assert np.isfinite(vec).all() and vec.shape[1] == 384
np.save(f'{OUT}/para_vectors_f16.npy', vec.astype('float16'))
df.to_parquet(f'{OUT}/para_chunks.parquet', index=False)
print('saved to', OUT, os.path.getsize(f'{OUT}/para_vectors_f16.npy') // 2**20, 'MB vectors')